In [ ]:
import os
import sys

import numpy as np
import pandas as pd


PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [ ]:
from src.database.database import connect_to_database

# Database connection

In [ ]:
cur, conn = connect_to_database()
print(cur, conn)

# Get data from bookable units table

- This table includes all products and information about the bookings for each day

In [ ]:
cur.execute('SELECT * FROM bookable_units')
data = cur.fetchall()

# Columns
column_names = [desc[0] for desc in cur.description]
print(column_names)

In [ ]:
df_bookable_units = pd.DataFrame(data, columns=column_names)
print(df_bookable_units.shape)

In [ ]:
df_bookable_units.head()

# Calculate price recommendation

Formula

- empfohlene Preisanpassung (in %) = 15% * (a - 0.5) * (0.5 + d)
- a = Momentane Auslastung (in %)
- d = (Tage bis zum Reiseantritt / 300)

Example
- Also wenn ein Produkt zum Beispiel 40 Tage vor Reiseantritt zu 75% ausgebucht ist, empfiehlt unsere Funktion eine Preisanpassung von 15% * (0.75 - 0.5) * (0.5 + (40 / 300)) = +2,375%

## Get current date

In [ ]:
# Convert date col to datetime for comparison later
df_bookable_units['date'] = pd.to_datetime(df_bookable_units['date'])

In [ ]:
current_date = pd.Timestamp.now().normalize()
print(current_date)

## Get max capacity

- Use maximum of count_available_bookings + count_optional_bookings for now
- A column with total capacity should be requested by customer

In [ ]:
df_bookable_units['total_available_bookings'] = (
    df_bookable_units['count_available_bookings']
    + df_bookable_units['count_optional_bookings']
)

df_bookable_units['max_capacity'] = df_bookable_units.groupby('product_id')[
    'total_available_bookings'
].transform('max')
df_bookable_units.head()

# Filter for future dates

In [ ]:
df_future = df_bookable_units[df_bookable_units['date'] >= current_date]
df_future.head()

In [ ]:
print(len(df_future))

## Calculate current occpancy

In [ ]:
occupancy_numerator = df_future['max_capacity'] - df_future['total_available_bookings']
df_future['occupancy_rate'] = np.where(
    df_future['max_capacity'] > 0, occupancy_numerator / df_future['max_capacity'], 0.0
)
df_future.head()

## Calculate days until travel start

In [ ]:
df_future['days_until_start'] = (df_future['date'] - current_date).dt.days
df_future['days_factor'] = df_future['days_until_start'] / 300

In [ ]:
df_future.head(10)

## Apply formula

- This df should be written to the db

In [ ]:
df_future['price_adjustment_pct'] = (
    0.15 * (df_future['occupancy_rate'] - 0.5) * (0.5 + df_future['days_factor'])
)

In [ ]:
df_future.head()

## Get max price adjustment

- This should be the output of the third endpoint

In [ ]:
max_price_adjustment = df_future['price_adjustment_pct'].max()

max_recommendation = df_future[
    df_future['price_adjustment_pct'] == max_price_adjustment
]
max_recommendation.iloc[0]